## Week 1: TEAM FORMATION & DATA HUNTING

Syllabus Mapping : Unit 1 Goals for this week:

Select the dataset (Google Play Store Apps - Kaggle)
Set up the GitHub repository for the project
Get comfortable with the two core libraries we will use throughout the project: Pandas and NumPy

Dataset source: https://www.kaggle.com/datasets/lava18/google-play-store-apps File used: googleplaystore.csv


In [ ]:
# Colab startup: fetch the project CSV when the runtime has no local copy.
from pathlib import Path
import pandas as pd
if not Path("googleplaystore.csv").exists():
    pd.read_csv("https://raw.githubusercontent.com/ran-nope/DSV_PROJECT/main/googleplaystore.csv").to_csv("googleplaystore.csv", index=False)

import pandas as pd
import numpy as np



# 2. LOADING THE DATASET FOR THE FIRST TIME

DATA_PATH = "googleplaystore.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape of the dataset (rows, columns):", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows of the dataset:")
print(df.head())

# 3. GETTING COMFORTABLE WITH PANDAS

print("\n--- Pandas basics ---")
print("Selecting a single column (App):")
print(df["App"].head())

print("\nSelecting multiple columns:")
print(df[["App", "Category", "Rating"]].head())

print("\nFiltering rows: apps with a Rating above 4.5")
high_rated = df[df["Rating"] > 4.5]
print(f"Number of apps rated above 4.5: {len(high_rated)}")

print("\nSorting apps by Rating (descending):")
print(df.sort_values(by="Rating", ascending=False)[["App", "Rating"]].head())

# 4. GETTING COMFORTABLE WITH NUMPY

print("\n--- NumPy basics ---")
ratings_array = df["Rating"].dropna().to_numpy()
print("Ratings converted to a NumPy array, sample:", ratings_array[:5])
print("Mean rating (NumPy):", np.mean(ratings_array))
print("Standard deviation of ratings (NumPy):", np.std(ratings_array))
print("Max rating:", np.max(ratings_array), "| Min rating:", np.min(ratings_array))

# A quick NumPy array operation demo
sample = np.array([1, 2, 3, 4, 5])
print("\nSample array:", sample)
print("Array squared:", sample ** 2)
print("Array mean:", sample.mean())

print("\nWeek 1 complete: dataset selected, repo initialised, "
      "and team is comfortable with Pandas/NumPy basics.")


## Week 2: "KNOW YOUR DATA"

Syllabus Mapping : Unit 1 (Practical) Goals for this week:

Load the dataset
Check data types of every column
Inspect the structure (shape, nulls, duplicates, uniques)
Generate a "First Impression" report summarising the dataset


In [ ]:
import pandas as pd

DATA_PATH = "googleplaystore.csv"
df = pd.read_csv(DATA_PATH)


# 1. BASIC STRUCTURE

print("=" * 60)
print("BASIC STRUCTURE")
print("=" * 60)
print("Shape (rows, columns):", df.shape)
print("\nColumn names:", df.columns.tolist())

# 2. CHECKING DATA TYPES

print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)
print(df.dtypes)

# Note: Reviews, Size, Installs and Price look numeric but are
# stored as text (object) because of characters like "+", ",",
# "M", "$". This is exactly what we will fix in Week 3.

# 3. df.info() AND df.describe()

print("\n" + "=" * 60)
print("df.info()")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("df.describe() - numeric columns")
print("=" * 60)
print(df.describe())

print("\n" + "=" * 60)
print("df.describe(include='object') - categorical columns")
print("=" * 60)
print(df.describe(include="object"))


# 4. MISSING VALUES

print("\n" + "=" * 60)
print("MISSING VALUES PER COLUMN")
print("=" * 60)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_report = pd.DataFrame({"missing_count": missing, "missing_%": missing_pct.round(2)})
print(missing_report[missing_report["missing_count"] > 0])

# 5. DUPLICATES

print("\n" + "=" * 60)
print("DUPLICATE ROWS")
print("=" * 60)
print("Number of fully duplicate rows:", df.duplicated().sum())
print("Number of duplicate App names:", df.duplicated(subset="App").sum())

# 6. UNIQUE VALUES IN KEY CATEGORICAL COLUMNS

print("\n" + "=" * 60)
print("UNIQUE VALUES - KEY COLUMNS")
print("=" * 60)
for col in ["Category", "Type", "Content Rating", "Genres"]:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts().head(10))

# 7. "FIRST IMPRESSION" REPORT

print("\n" + "=" * 60)
print("FIRST IMPRESSION REPORT")
print("=" * 60)
report = f"""
Dataset          : Google Play Store Apps
Total records    : {df.shape[0]}
Total features   : {df.shape[1]}
Numeric columns  : {df.select_dtypes(include='number').columns.tolist()}
Text columns     : {df.select_dtypes(include='object').columns.tolist()}
Columns with missing data : {missing[missing > 0].index.tolist()}
Duplicate rows   : {df.duplicated().sum()}
Most common category : {df['Category'].mode()[0]}
Free vs Paid split : {df['Type'].value_counts().to_dict()}
Observations:
 - Reviews, Size, Installs, Price are stored as text and need cleaning.
 - Rating has missing values that will need imputation.
 - Category, Genres, Content Rating are categorical and can be
   used as grouping variables for later analysis.
"""
print(report)

with open("first_impression_report.txt", "w") as f:
    f.write(report)
print("Saved report to first_impression_report.txt")


## Week 3: THE CLEANING SPRINT

Syllabus Mapping : Unit 2 Goals for this week (mapped to Unit 2 - Descriptive Statistics):

Convert messy text columns (Reviews, Size, Installs, Price) into proper numeric columns
Compute Measures of Central Tendency (Mean, Median, Mode) and Measures of Dispersion (Range, Variance, Std Dev, IQR) + Skewness/Kurtosis - this is what decides HOW we impute
Visualize missing data (missingness matrix / heatmap) before handling it, then apply imputation strategies
Detect outliers using Boxplots, the IQR method, and Z-score
Save a cleaned dataset for next week's EDA


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")

DATA_PATH = "googleplaystore.csv"
df = pd.read_csv(DATA_PATH)

print("Shape before cleaning:", df.shape)
# Reject malformed or shifted records before any imputation.
valid_rating = pd.to_numeric(df["Rating"], errors="coerce")
valid_installs = pd.to_numeric(df["Installs"].astype(str).str.replace(r"[,+]", "", regex=True), errors="coerce")
valid_price = pd.to_numeric(df["Price"].astype(str).str.replace("$", "", regex=False), errors="coerce")
valid_reviews = pd.to_numeric(df["Reviews"], errors="coerce")
valid = (valid_rating.isna() | valid_rating.between(1, 5)) & valid_installs.ge(0) & valid_price.ge(0) & valid_reviews.ge(0)
df.loc[~valid].to_csv("week3_quarantined_records.csv", index=False)
df = df.loc[valid].copy()

# 1. REMOVE DUPLICATES

before = df.shape[0]
df = df.drop_duplicates()
print(f"Removed {before - df.shape[0]} duplicate rows.")


# 2. CLEANING "Installs" -> numeric
#    e.g. "10,000+"  ->  10000

df["Installs"] = (
    df["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)

)
df["Installs"] = pd.to_numeric(df["Installs"], errors="coerce")


# 3. CLEANING "Price" -> numeric
#    e.g. "$4.99" -> 4.99 , "0" -> 0.0

df["Price"] = df["Price"].astype(str).str.replace("$", "", regex=False)
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

# 4. CLEANING "Size" -> numeric (in MB)
#    e.g. "19M" -> 19.0 , "Varies with device" -> NaN (imputed later)

def clean_size(value):
    value = str(value)
    if "Varies with device" in value:
        return np.nan
    if value.endswith("M"):
        return float(value.replace("M", ""))
    if value.endswith("k"):
        return float(value.replace("k", "")) / 1024   # convert KB to MB
    return np.nan

df["Size"] = df["Size"].apply(clean_size)

# 5. CLEANING "Reviews" -> numeric

df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")

print("\nData types after cleaning:")
print(df[["Installs", "Price", "Size", "Reviews", "Rating"]].dtypes)

# 6. DESCRIPTIVE STATISTICS: CENTRAL TENDENCY & DISPERSION
#    Mean/Median/Mode, Range/Variance/Std Dev/IQR,Skewness & Kurtosis


numeric_cols = ["Rating", "Reviews", "Size", "Installs", "Price"]

print("\n" + "=" * 60)
print("DESCRIPTIVE STATISTICS (before imputation)")
print("=" * 60)
desc_rows = []
for col in numeric_cols:
    series = df[col].dropna()
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    desc_rows.append({
        "Column": col,
        "Mean": round(series.mean(), 2),
        "Median": round(series.median(), 2),
        "Mode": round(series.mode()[0], 2) if not series.mode().empty else np.nan,
        "Range": round(series.max() - series.min(), 2),
        "Variance": round(series.var(), 2),
        "Std Dev": round(series.std(), 2),
        "IQR": round(q3 - q1, 2),
        "Skewness": round(series.skew(), 2),
        "Kurtosis": round(series.kurt(), 2),
    })
desc_df = pd.DataFrame(desc_rows)
print(desc_df.to_string(index=False))
print(
    "\nReading the skewness column: values near 0 = roughly symmetric, "
    "positive = right-skewed (long tail of high values), negative = "
    "left-skewed. This is exactly why we use the MEDIAN (not the mean) "
    "to impute skewed columns like Rating and Size below."
)


# 7. MISSING DATA VISUALIZATION (before imputation)
#    Missingness Matrix / Heatmap

print("\nMissing values BEFORE imputation:")
print(df[["Rating", "Size", "Type"]].isnull().sum())

plt.figure(figsize=(10, 5))
try:
    import missingno as msno
    msno.matrix(df[["Rating", "Reviews", "Size", "Installs", "Price", "Type"]])
    plt.title("Missingness Matrix (missingno)")
except ImportError:
    # Fallback if missingno isn't installed: a boolean heatmap does the
    # same job - yellow/light cells mark a missing value in that column.
    sns.heatmap(
        df[["Rating", "Reviews", "Size", "Installs", "Price", "Type"]].isnull(),
        cbar=False, cmap="viridis"
    )
    plt.title("Missing Data Heatmap (True = missing)")
plt.tight_layout()
plt.savefig("week3_missing_data_matrix.png", dpi=150)
plt.show()
plt.close()
print("Saved missingness visualization to week3_missing_data_matrix.png")
print(
    "Missingness visualization is descriptive only. "
    "It cannot establish MCAR. "
    "Median/mode imputation is a baseline; final-review summaries preserve observed ratings."
)


# 8. HANDLING MISSING VALUES (Imputation)

# Rating -> numeric, skewed -> use MEDIAN (robust to outliers/skew)
df["Rating"] = df["Rating"].fillna(df["Rating"].median())

# Size -> numeric -> use MEDIAN as well ("Varies with device" left many NaNs)
df["Size"] = df["Size"].fillna(df["Size"].median())

# Type -> categorical -> use MODE (most frequent category)
df["Type"] = df["Type"].fillna(df["Type"].mode()[0])

print("\nMissing values AFTER imputation:")
print(df[["Rating", "Size", "Type"]].isnull().sum())


# 9. OUTLIER DETECTION - BOXPLOTS (visual identification, Unit 2)

fig, axes = plt.subplots(1, len(numeric_cols), figsize=(20, 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(y=df[col], ax=ax, color="skyblue")
    ax.set_title(f"Boxplot: {col}")
plt.tight_layout()
plt.savefig("week3_boxplots.png", dpi=150)
plt.show()
plt.close()
print("\nSaved boxplots to week3_boxplots.png")


# 10. OUTLIER DETECTION - IQR METHOD
#     outliers are below Q1 - 1.5*IQR or above Q3 + 1.5*IQR

print("\nOutlier detection using the IQR method (Q1 - 1.5*IQR, Q3 + 1.5*IQR):")
iqr_summary = {}
for col in numeric_cols:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)].shape[0]
    iqr_summary[col] = n_outliers
    print(f"  {col}: bounds=({lower_bound:.2f}, {upper_bound:.2f}) -> {n_outliers} outliers")

# 11. OUTLIER DETECTION - Z-SCORE METHOD (cross-check against IQR)

print("\nOutlier detection using Z-score (|z| > 3):")
z_summary = {}
for col in numeric_cols:
    z_scores = np.abs(stats.zscore(df[col].dropna()))
    n_outliers = (z_scores > 3).sum()
    z_summary[col] = n_outliers
    print(f"  {col}: {n_outliers} potential outliers")

print(
    "\nNote: IQR usually flags more borderline points than Z-score on "
    "skewed columns like Installs/Price - this is expected and worth "
    "mentioning in the report as a comparison of the two methods."
)

# Example: capping Reviews outliers at the 99th percentile
# a common strategy so that a few viral apps don't distort later analysis
upper_cap = df["Reviews"].quantile(0.99)
df["Reviews_capped_for_plot"] = np.minimum(df["Reviews"], upper_cap)
print(f"\nCreated a separate plot-only Reviews column capped at the 99th percentile ({upper_cap:.0f}).")

# 9. SAVE CLEANED DATASET

df.to_csv("googleplaystore_cleaned.csv", index=False)
print("\nCleaned dataset saved as googleplaystore_cleaned.csv")
print("Final shape:", df.shape)


## Week 4 - EDA Deep Dive (Unit 2 Practical)
Univariate and bivariate analysis of the cleaned Play Store data: ratings,
categories, installs, Free vs Paid, and a correlation heatmap.
Run after week3_cleaning_sprint.py. Saves week4_*.png and week4_insights.txt.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

df = pd.read_csv("googleplaystore_cleaned.csv")
print("Loaded cleaned dataset. Shape:", df.shape)

# PART A : UNIVARIATE ANALYSIS

# A1. Distribution of Rating
plt.figure(figsize=(8, 5))
sns.histplot(df["Rating"], bins=20, kde=True, color="steelblue")
plt.title("Distribution of App Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Apps")
plt.tight_layout()
plt.savefig("week4_univariate_rating.png", dpi=150)
plt.show()
plt.close()
print("Saved: week4_univariate_rating.png")

# A2. Number of apps per Category
plt.figure(figsize=(10, 5))
order = df["Category"].value_counts().index
sns.countplot(y="Category", data=df, order=order, hue="Category", legend=False, palette="viridis")
plt.title("Number of Apps per Category")
plt.xlabel("Count")
plt.ylabel("Category")
plt.tight_layout()
plt.savefig("week4_univariate_category.png", dpi=150)
plt.show()
plt.close()
print("Saved: week4_univariate_category.png")

# A3. Free vs Paid split
plt.figure(figsize=(5, 5))
df["Type"].value_counts().plot.pie(autopct="%1.1f%%", colors=["#66b3ff", "#ff9999"])
plt.title("Free vs Paid Apps")
plt.ylabel("")
plt.tight_layout()
plt.savefig("week4_univariate_type.png", dpi=150)
plt.show()
plt.close()
print("Saved: week4_univariate_type.png")

# A4. Distribution of Installs (log scale, since installs vary hugely)
plt.figure(figsize=(8, 5))
sns.histplot(np.log10(df["Installs"].replace(0, 1)), bins=20, color="seagreen")
plt.title("Distribution of Installs (log10 scale)")
plt.xlabel("log10(Installs)")
plt.ylabel("Number of Apps")
plt.tight_layout()
plt.savefig("week4_univariate_installs.png", dpi=150)
plt.show()
plt.close()
print("Saved: week4_univariate_installs.png")

# A5. Univariate Boxplot - Rating ( Box Plot = univariate,
#     used for outlier detection, distinct from the bivariate boxplot
#     in Part B below
plt.figure(figsize=(4, 5))
sns.boxplot(y=df["Rating"], color="lightcoral")
plt.title("Univariate Boxplot: Rating")
plt.tight_layout()
plt.savefig("week4_univariate_rating_boxplot.png", dpi=150)
plt.show()
plt.close()
print("Saved: week4_univariate_rating_boxplot.png")

# PART B : BIVARIATE ANALYSIS

# B1. Rating vs Category (Boxplot)
plt.figure(figsize=(12, 6))
sns.boxplot(x="Category", y="Rating", data=df, hue="Category", legend=False, palette="Set2")
plt.title("Rating Distribution across Categories")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("week4_bivariate_rating_category.png", dpi=150)
plt.show()
plt.close()
print("Saved: week4_bivariate_rating_category.png")

# B2. Reviews vs Installs (scatter, log-log since both are skewed)
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=np.log10(df["Installs"].replace(0, 1)),
    y=np.log10(df["Reviews"].replace(0, 1)),
    hue=df["Type"],
    alpha=0.6,
)
plt.title("Reviews vs Installs (log-log scale)")
plt.xlabel("log10(Installs)")
plt.ylabel("log10(Reviews)")
plt.tight_layout()
plt.savefig("week4_bivariate_reviews_installs.png", dpi=150)
plt.show()
plt.close()
print("Saved: week4_bivariate_reviews_installs.png")

# B3. Rating vs Type (Free vs Paid)
plt.figure(figsize=(6, 5))
sns.violinplot(x="Type", y="Rating", data=df, hue="Type", legend=False, palette="pastel")
plt.title("Rating Distribution: Free vs Paid Apps")
plt.tight_layout()
plt.savefig("week4_bivariate_rating_type.png", dpi=150)
plt.show()
plt.close()
print("Saved: week4_bivariate_rating_type.png")

# B4. Correlation heatmap of numeric features
plt.figure(figsize=(7, 6))
numeric_df = df[["Rating", "Reviews", "Size", "Installs", "Price"]]
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
plt.savefig("week4_correlation_heatmap.png", dpi=150)
plt.show()
plt.close()
print("Saved: week4_correlation_heatmap.png")

# PART C : KEY INSIGHTS SUMMARY

insights = f"""
WEEK 4 - KEY EDA INSIGHTS
--------------------------------------------------
1. Average app rating is {df['Rating'].mean():.2f}, and most ratings
   cluster between 4.0 and 4.5 - users tend to rate apps generously.
2. {df['Category'].value_counts().idxmax()} is the most common category
   in this sample ({df['Category'].value_counts().max()} apps).
3. {(df['Type'].value_counts(normalize=True)['Free']*100):.1f}% of apps
   are Free, confirming a freemium-dominated market.
4. Installs and Reviews show a positive relationship on the log scale -
   apps with more installs tend to accumulate more reviews.
5. Correlation between Rating and other numeric features
   (Reviews, Size, Price, Installs) is generally weak, suggesting
   rating alone is not driven by any single numeric factor - this is
   part of the motivation for the PCA step planned in later weeks.
"""
print(insights)
with open("week4_insights.txt", "w") as f:
    f.write(insights)
print("Saved: week4_insights.txt")


## Week 6 - Feature Engineering (Unit 3)
Fixes the skew in Reviews, Installs, Size and Price with log1p, scales the
numeric columns, and encodes Type, Content Rating and Category.
Run after week3_cleaning_sprint.py. Saves googleplaystore_engineered.csv.


In [ ]:
"""
WEEK 6 : FEATURE ENGINEERING
Syllabus Mapping : Unit 3
----------------------------------------------------------------
Goals for this week:
1. Measure skewness and fix it with log transformations
2. Apply feature scaling (StandardScaler and MinMaxScaler)
3. Encode categorical variables (Label + One-Hot)
4. Show visual proof (Before vs After histograms/KDEs)
5. Save an engineered dataset for Week 7 (PCA)

Input  : googleplaystore_cleaned.csv   (created by week3_cleaning_sprint.py)
Output : googleplaystore_engineered.csv, week6_*.png, week6_feature_report.txt
----------------------------------------------------------------
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

sns.set_style("whitegrid")

DATA_PATH = "googleplaystore_cleaned.csv"
df = pd.read_csv(DATA_PATH)
print("Loaded cleaned data:", df.shape)

# ----------------------------------------------------------------
# 0. SAFETY CHECKS
# ----------------------------------------------------------------
# The raw Play Store file has one shifted row (Rating = 19, Category = "1.9").
# Ratings must lie between 1 and 5, so anything else is a data-entry error.
bad_rows = (~df["Rating"].between(1, 5)).sum()
df = df[df["Rating"].between(1, 5)].copy()
print(f"Dropped {bad_rows} row(s) with an impossible Rating value.")

# ----------------------------------------------------------------
# 1. DATE FEATURE : how stale is each app?
# ----------------------------------------------------------------
df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors="coerce")
reference_date = df["Last Updated"].max()
df["days_since_update"] = (reference_date - df["Last Updated"]).dt.days
df["days_since_update"] = df["days_since_update"].fillna(df["days_since_update"].median())

# ----------------------------------------------------------------
# 2. SKEWNESS ANALYSIS (decides which columns need a transformation)
# ----------------------------------------------------------------
numeric_cols = ["Rating", "Reviews", "Size", "Installs", "Price", "days_since_update"]
skew_before = df[numeric_cols].skew().round(2)
print("\nSkewness BEFORE transformation:")
print(skew_before.to_string())

# Rule of thumb: |skew| > 1 is highly skewed
skewed_cols = skew_before[skew_before.abs() > 1].index.tolist()
skewed_cols = [c for c in skewed_cols if c != "Rating"]  # Rating is bounded 1-5
print("\nColumns selected for log transformation:", skewed_cols)

# ----------------------------------------------------------------
# 3. FIXING SKEWNESS : log(1 + x)
#    log1p is used instead of log because Price and Reviews contain zeros
# ----------------------------------------------------------------
for col in skewed_cols:
    df[f"log_{col}"] = np.log1p(df[col])

log_cols = [f"log_{c}" for c in skewed_cols]
skew_after = df[log_cols].skew().round(2)
print("\nSkewness AFTER transformation:")
print(skew_after.to_string())

# ----------------------------------------------------------------
# 4. VISUAL PROOF : Before vs After histograms with KDE
# ----------------------------------------------------------------
fig, axes = plt.subplots(2, len(skewed_cols), figsize=(5 * len(skewed_cols), 8))
axes = np.atleast_2d(axes)
for i, col in enumerate(skewed_cols):
    sns.histplot(df[col], kde=True, ax=axes[0, i], color="#E4572E", bins=40)
    axes[0, i].set_title(f"{col} (before)\nskew = {skew_before[col]}")
    sns.histplot(df[f"log_{col}"], kde=True, ax=axes[1, i], color="#17BEBB", bins=40)
    axes[1, i].set_title(f"log1p({col}) (after)\nskew = {skew_after[f'log_{col}']}")
plt.suptitle("Fixing Skewness: Before vs After Log Transformation", fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig("week6_skewness_before_after.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()

# ----------------------------------------------------------------
# 5. FEATURE SCALING : StandardScaler vs MinMaxScaler
# ----------------------------------------------------------------
scale_cols = ["Rating"] + log_cols + ["days_since_update"]
scale_cols = [c for c in scale_cols if c in df.columns]

std_scaler = StandardScaler()
mm_scaler = MinMaxScaler()

std_vals = std_scaler.fit_transform(df[scale_cols])
mm_vals = mm_scaler.fit_transform(df[scale_cols])

for j, col in enumerate(scale_cols):
    df[f"{col}_std"] = std_vals[:, j]
    df[f"{col}_mm"] = mm_vals[:, j]

print("\nScaled feature summary (StandardScaler: mean ~ 0, std ~ 1):")
print(pd.DataFrame(std_vals, columns=scale_cols).describe().loc[["mean", "std"]].round(3).to_string())
print("\nScaled feature summary (MinMaxScaler: range 0 to 1):")
print(pd.DataFrame(mm_vals, columns=scale_cols).describe().loc[["min", "max"]].round(3).to_string())

# Compare one column across the three versions
demo = log_cols[0] if log_cols else "Rating"
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df[demo], kde=True, ax=axes[0], color="#6C757D")
axes[0].set_title(f"{demo} (original scale)")
sns.histplot(df[f"{demo}_std"], kde=True, ax=axes[1], color="#2E86AB")
axes[1].set_title(f"{demo} (StandardScaler)")
sns.histplot(df[f"{demo}_mm"], kde=True, ax=axes[2], color="#F18F01")
axes[2].set_title(f"{demo} (MinMaxScaler)")
plt.tight_layout()
plt.savefig("week6_scaling_comparison.png", dpi=150)
plt.show()
plt.close()

# ----------------------------------------------------------------
# 6. ENCODING CATEGORICAL VARIABLES
# ----------------------------------------------------------------
# 6a. Label encoding : Type is binary (Free / Paid)
le = LabelEncoder()
df["Type_enc"] = le.fit_transform(df["Type"])
print("\nLabel encoding for Type:", {c: int(i) for i, c in enumerate(le.classes_)})

# 6b. One-hot encoding : Content Rating has a handful of unordered categories
df = pd.get_dummies(df, columns=["Content Rating"], prefix="CR", dtype=int)

# 6c. One-hot on Category : 33 categories are too many, so keep the top 10
#     and group the rest as "Other" to avoid a huge sparse matrix
top_categories = df["Category"].value_counts().head(10).index
df["Category_grouped"] = np.where(df["Category"].isin(top_categories), df["Category"], "OTHER")
df = pd.get_dummies(df, columns=["Category_grouped"], prefix="Cat", dtype=int)

print("Top 10 categories kept for one-hot encoding:")
print(list(top_categories))

# ----------------------------------------------------------------
# 7. SAVE FEATURE REPORT + ENGINEERED DATASET
# ----------------------------------------------------------------
with open("week6_feature_report.txt", "w", encoding="utf-8") as f:
    f.write("WEEK 6 FEATURE ENGINEERING REPORT\n")
    f.write("=" * 50 + "\n\n")
    f.write("Skewness before transformation:\n" + skew_before.to_string() + "\n\n")
    f.write("Skewness after log1p transformation:\n" + skew_after.to_string() + "\n\n")
    f.write(f"Scaled columns: {scale_cols}\n\n")
    f.write("Label encoded: Type -> " + str({str(c): int(i) for i, c in enumerate(le.classes_)}) + "\n")
    f.write("One-hot encoded: Content Rating, Category (top 10 + OTHER)\n")

df.to_csv("googleplaystore_engineered.csv", index=False)
print("\nSaved: googleplaystore_engineered.csv, week6_feature_report.txt")
print("Final shape:", df.shape)

## Week 7 - Dimensionality Reduction (Unit 4)
Drops highly correlated features (|r| > 0.85), runs PCA on what's left,
and plots the explained variance and the first components in 2D and 3D.
Run after week6_feature_engineering.py. Saves googleplaystore_pca.csv.


In [ ]:
"""
WEEK 7 : DIMENSIONALITY REDUCTION
Syllabus Mapping : Unit 4
----------------------------------------------------------------
Goals for this week:
1. Feature selection: correlation matrix + removal of multicollinear features
2. PCA on the standardised feature space
3. Visualise the Explained Variance Ratio (scree + cumulative)
4. 2D and 3D scatter plots of the Principal Components
5. Interpret components with a loadings heatmap

Input  : googleplaystore_engineered.csv   (created by week6_feature_engineering.py)
Output : googleplaystore_pca.csv, week7_*.png, week7_pca_report.txt
----------------------------------------------------------------
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3D projection)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_style("whitegrid")

DATA_PATH = "googleplaystore_engineered.csv"
df = pd.read_csv(DATA_PATH)
print("Loaded engineered data:", df.shape)

# ----------------------------------------------------------------
# 1. CANDIDATE FEATURES
#    PCA works on continuous variance, so we use the transformed numeric
#    features plus the binary Type flag. One-hot columns (Category, Content
#    Rating) are kept out of PCA because they would swamp the variance with
#    sparse 0/1 dummies; they are used later to colour the scatter plots.
# ----------------------------------------------------------------
numeric_features = [c for c in df.columns if c.startswith("log_") and not c.endswith(("_std", "_mm"))]
candidate_features = ["Rating", "days_since_update", "Type_enc"] + numeric_features
candidate_features = [c for c in candidate_features if c in df.columns]
print(f"\nCandidate features ({len(candidate_features)}):")
print(candidate_features)

# ----------------------------------------------------------------
# 2. CORRELATION MATRIX
# ----------------------------------------------------------------
corr = df[candidate_features].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, cbar_kws={"label": "Pearson r"})
plt.title("Correlation Matrix (before feature selection)")
plt.tight_layout()
plt.savefig("week7_correlation_matrix.png", dpi=150)
plt.show()
plt.close()

# ----------------------------------------------------------------
# 3. FEATURE SELECTION : drop one feature from each highly correlated pair
# ----------------------------------------------------------------
THRESHOLD = 0.85
corr_all = df[candidate_features].corr().abs()
upper = corr_all.where(np.triu(np.ones(corr_all.shape), k=1).astype(bool))

to_drop = []
print(f"\nPairs with |r| > {THRESHOLD} (multicollinearity):")
for col in upper.columns:
    for row in upper.index:
        val = upper.loc[row, col]
        if pd.notna(val) and val > THRESHOLD and col not in to_drop:
            print(f"  {row}  <->  {col}   r = {val:.2f}  -> dropping '{col}'")
            to_drop.append(col)

selected_features = [c for c in candidate_features if c not in to_drop]
print(f"\nDropped {len(to_drop)} feature(s): {to_drop}")
print(f"Kept {len(selected_features)} feature(s) for PCA.")

# ----------------------------------------------------------------
# 4. STANDARDISE (PCA is variance-based, so scale matters)
# ----------------------------------------------------------------
X = df[selected_features].astype(float)
X_scaled = StandardScaler().fit_transform(X)

# ----------------------------------------------------------------
# 5. PCA : full decomposition first, to study explained variance
# ----------------------------------------------------------------
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

evr = pca_full.explained_variance_ratio_
cum_evr = np.cumsum(evr)
n_95 = int(np.argmax(cum_evr >= 0.95) + 1)
n_80 = int(np.argmax(cum_evr >= 0.80) + 1)
print(f"\nComponents needed for 80% variance: {n_80}")
print(f"Components needed for 95% variance: {n_95}  (from {len(selected_features)} features)")

# ----------------------------------------------------------------
# 6. EXPLAINED VARIANCE RATIO PLOT (scree + cumulative)
# ----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(1, len(evr) + 1)
ax.bar(x, evr, color="#2E86AB", alpha=0.85, label="Individual")
ax.plot(x, cum_evr, color="#E4572E", marker="o", linewidth=2, label="Cumulative")
ax.axhline(0.95, color="grey", linestyle="--", linewidth=1)
ax.axvline(n_95, color="grey", linestyle="--", linewidth=1)
ax.text(len(evr) * 0.55, 0.96, f"95% variance at PC{n_95}", color="grey")
ax.set_xlabel("Principal Component")
ax.set_ylabel("Explained Variance Ratio")
ax.set_title("PCA Explained Variance Ratio")
ax.set_xticks(x)
ax.legend(loc="center right")
plt.tight_layout()
plt.savefig("week7_explained_variance.png", dpi=150)
plt.show()
plt.close()

# ----------------------------------------------------------------
# 7. REDUCED PCA (3 components for plotting, n_95 for the reduced dataset)
# ----------------------------------------------------------------
pca3 = PCA(n_components=3, random_state=42)
pcs = pca3.fit_transform(X_scaled)
df["PC1"], df["PC2"], df["PC3"] = pcs[:, 0], pcs[:, 1], pcs[:, 2]
print("\nVariance captured by PC1, PC2, PC3:",
      [f"{v:.1%}" for v in pca3.explained_variance_ratio_],
      f"(total {pca3.explained_variance_ratio_.sum():.1%})")

# ----------------------------------------------------------------
# 8. 2D SCATTER OF PRINCIPAL COMPONENTS
# ----------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(data=df, x="PC1", y="PC2", hue="Type", alpha=0.5, s=15,
                palette={"Free": "#2E86AB", "Paid": "#E4572E"}, ax=axes[0])
axes[0].set_title("PC1 vs PC2 by App Type")
axes[0].set_xlabel(f"PC1 ({pca3.explained_variance_ratio_[0]:.1%})")
axes[0].set_ylabel(f"PC2 ({pca3.explained_variance_ratio_[1]:.1%})")

top_cats = df["Category"].value_counts().head(5).index
sub = df[df["Category"].isin(top_cats)]
sns.scatterplot(data=sub, x="PC1", y="PC2", hue="Category", alpha=0.5, s=15,
                palette="Set2", ax=axes[1])
axes[1].set_title("PC1 vs PC2 for the Top 5 Categories")
axes[1].set_xlabel(f"PC1 ({pca3.explained_variance_ratio_[0]:.1%})")
axes[1].set_ylabel(f"PC2 ({pca3.explained_variance_ratio_[1]:.1%})")

plt.tight_layout()
plt.savefig("week7_pca_2d_scatter.png", dpi=150)
plt.show()
plt.close()

# ----------------------------------------------------------------
# 9. 3D SCATTER OF PRINCIPAL COMPONENTS
# ----------------------------------------------------------------
sample = df.sample(n=min(3000, len(df)), random_state=42)
colors = sample["Type"].map({"Free": "#2E86AB", "Paid": "#E4572E"})

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(sample["PC1"], sample["PC2"], sample["PC3"], c=colors, alpha=0.5, s=10)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("First Three Principal Components (3,000 sampled apps)")
plt.tight_layout()
plt.savefig("week7_pca_3d_scatter.png", dpi=150)
plt.show()
plt.close()

# ----------------------------------------------------------------
# 10. LOADINGS HEATMAP : what does each component actually mean?
# ----------------------------------------------------------------
loadings = pd.DataFrame(
    pca3.components_.T,
    index=selected_features,
    columns=["PC1", "PC2", "PC3"],
)
# keep the strongest contributors so the heatmap stays readable
top_idx = loadings.abs().max(axis=1).sort_values(ascending=False).head(10).index

plt.figure(figsize=(8, 8))
sns.heatmap(loadings.loc[top_idx], annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("PCA Loadings (top 10 features)")
plt.tight_layout()
plt.savefig("week7_pca_loadings.png", dpi=150)
plt.show()
plt.close()

print("\nTop drivers of each component:")
for pc in ["PC1", "PC2", "PC3"]:
    top = loadings[pc].abs().sort_values(ascending=False).head(3).index.tolist()
    print(f"  {pc}: {top}")

# ----------------------------------------------------------------
# 11. SAVE REDUCED DATASET + REPORT
# ----------------------------------------------------------------
pca_n = PCA(n_components=n_95, random_state=42)
reduced = pca_n.fit_transform(X_scaled)
reduced_df = pd.DataFrame(reduced, columns=[f"PC{i + 1}" for i in range(n_95)])
reduced_df.insert(0, "App", df["App"].values)
reduced_df.insert(1, "Category", df["Category"].values)
reduced_df.insert(2, "Type", df["Type"].values)
reduced_df.to_csv("googleplaystore_pca.csv", index=False)

with open("week7_pca_report.txt", "w", encoding="utf-8") as f:
    f.write("WEEK 7 DIMENSIONALITY REDUCTION REPORT\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Correlation threshold: {THRESHOLD}\n")
    f.write(f"Features dropped for multicollinearity: {to_drop}\n")
    f.write(f"Features kept: {len(selected_features)}\n\n")
    f.write("Explained variance ratio per component:\n")
    for i, (e, c) in enumerate(zip(evr, cum_evr), start=1):
        f.write(f"  PC{i}: {e:.4f}   cumulative: {c:.4f}\n")
    f.write(f"\nComponents for 95% variance: {n_95}\n")

print(f"\nSaved: googleplaystore_pca.csv ({n_95} components), week7_pca_report.txt")

In [ ]:
# Reproducibility check: execute the actual GitHub scripts twice in isolation.
from pathlib import Path
from urllib.request import urlopen
import os, io, json, contextlib, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/ran-nope/DSV_PROJECT/main/"
SCRIPTS = ["week1_data_hunting.py", "week2_know_your_data.py", "week3_cleaning_sprint.py",
           "week4_eda_deep_dive.py", "week6_feature_engineering.py", "week7_dimensionality_reduction.py"]
root = Path.cwd()
check_dir = root / "validation_runs"
check_dir.mkdir(exist_ok=True)
raw_bytes = urlopen(BASE + "googleplaystore.csv").read()
sources = {name: urlopen(BASE + name).read().decode("utf-8-sig") for name in SCRIPTS}
for name, source in sources.items():
    compile(source, name, "exec")
print("SYNTAX CHECKS PASSED: all six GitHub scripts")
original_show = plt.show
plt.show = lambda *args, **kwargs: None
frames = []
try:
    for run in [1, 2]:
        run_dir = check_dir / f"run_{run}"
        run_dir.mkdir(exist_ok=True)
        (run_dir / "googleplaystore.csv").write_bytes(raw_bytes)
        os.chdir(run_dir)
        namespaces = {}
        for name, source in sources.items():
            ns = {"__name__": "__main__", "__file__": name}
            log = io.StringIO()
            with contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
                exec(compile(source, name, "exec"), ns)
            Path(name + ".log").write_text(log.getvalue(), encoding="utf-8")
            namespaces[name] = ns
            print(f"RUN {run}: {name} completed")
        cleaned = pd.read_csv("googleplaystore_cleaned.csv")
        engineered = pd.read_csv("googleplaystore_engineered.csv")
        reduced = pd.read_csv("googleplaystore_pca.csv")
        assert len(cleaned) == len(engineered) == len(reduced) == 10357
        assert cleaned["Rating"].between(1, 5).all()
        assert cleaned[["Reviews", "Size", "Installs", "Price"]].notna().all().all()
        assert cleaned[["Reviews", "Size", "Installs", "Price"]].ge(0).all().all()
        assert not cleaned.duplicated().any()
        assert "Life Made WI-Fi Touchscreen Photo Frame" not in cleaned["App"].values
        w6 = namespaces[SCRIPTS[-2]]
        w7 = namespaces[SCRIPTS[-1]]
        assert np.isfinite(w7["X_scaled"]).all()
        assert np.allclose(w6["std_vals"].mean(axis=0), 0, atol=1e-9)
        assert np.all(w6["mm_vals"] >= -1e-9) and np.all(w6["mm_vals"] <= 1+1e-9)
        assert np.isclose(w7["evr"].sum(), 1)
        assert np.all(np.diff(w7["cum_evr"]) >= 0)
        assert w7["pca_n"].explained_variance_ratio_.sum() >= .95
        assert np.allclose(w7["pca_full"].inverse_transform(w7["pca_full"].transform(w7["X_scaled"])), w7["X_scaled"])
        assert len([c for c in reduced if c.startswith("PC")]) == w7["n_95"]
        for prefix in ["CR_", "Cat_"]:
            assert engineered.filter(regex="^" + prefix).sum(axis=1).eq(1).all()
        report = Path("week6_feature_report.txt").read_text()
        assert "'Free': 0" in report and "'Paid': 1" in report
        expected_images = ["week6_skewness_before_after.png", "week6_scaling_comparison.png",
            "week7_correlation_matrix.png", "week7_explained_variance.png",
            "week7_pca_2d_scatter.png", "week7_pca_3d_scatter.png", "week7_pca_loadings.png"]
        assert all(Path(p).stat().st_size > 1000 for p in expected_images)
        frames.append((engineered, reduced))
        print(f"RUN {run}: data, scaling, encoding, PCA and seven image checks PASSED")
finally:
    os.chdir(root)
    plt.show = original_show
pd.testing.assert_frame_equal(frames[0][0], frames[1][0])
pd.testing.assert_frame_equal(frames[0][1], frames[1][1])
summary = {"status": "PASS", "runs": 2, "rows": len(frames[0][0]),
    "engineered_columns": len(frames[0][0].columns), "pca_components_95pct": w7["n_95"],
    "selected_features": w7["selected_features"], "dropped_features": w7["to_drop"],
    "explained_variance": w7["evr"].tolist(),
    "source_sha256": {name: hashlib.sha256(source.encode()).hexdigest() for name, source in sources.items()}}
(check_dir / "validation_summary.json").write_text(json.dumps(summary, indent=2))
print("REPEATABILITY PASSED: both runs produced identical engineered and PCA datasets")
print(json.dumps(summary, indent=2))


In [ ]:
# Gallery from the independently validated GitHub pipeline
from IPython.display import Image, HTML, display
print("Verified PCA explained variance | 10,357 records | 5 components retain 95.08%")
display(Image(filename="validation_runs/run_2/week7_explained_variance.png", width=1000))
print("Verified 2D PCA projection | App type and top categories")
display(Image(filename="validation_runs/run_2/week7_pca_2d_scatter.png", width=1000))
display(HTML(f'''<div style="max-width:360px;padding:22px;background:#101c30;color:#e6eefb;border-radius:12px;font-family:Arial,sans-serif">
<div style="font-size:12px;color:#58d9ba;letter-spacing:2px">DSV PROJECT / VERIFIED RUN</div>
<h2 style="color:white;font-size:23px;margin:12px 0">Reproducibility: PASS</h2>
<p>Two isolated runs of all six GitHub scripts produced identical exports.</p>
<p><b>{summary['rows']:,}</b> records &nbsp; / &nbsp; <b>{summary['engineered_columns']}</b> columns</p>
<p><b>{summary['pca_components_95pct']}</b> principal components retain <b>{100*sum(summary['explained_variance'][:summary['pca_components_95pct']]):.2f}%</b> variance</p>
<p style="color:#58d9ba">Data integrity · Scaling · Encoding<br>PCA reconstruction · Chart exports</p>
<div style="font-size:11px;color:#a8bad2">Validated 3 November 2026 · Python 3 / Colab</div>
</div>'''))
